# M7 — learned A* heuristic (Marswalk)

Trains a small network to predict on-foot **cost-to-go to a goal**, using training data generated by Marswalk's *own* planner (no external dataset). It's tiny — runs in a few minutes on **any GPU or even CPU**.

**How to run:** Runtime → GPU (optional), then **Run all**. When it's done, download **`heuristic_report.json`** and **`heuristic_samples.png`** and send them back. That's it.

In [ ]:
!pip install -q "git+https://github.com/MONSTER4REX/Martian-Map.git"
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', dev,
      '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')

## Build data from the planner, train, and evaluate

Targets are exact `time_to_haven_field` computations, so nothing here is fabricated. We report error vs the trivial straight-line lower bound and the **admissible fraction** (share where prediction ≤ truth — what A* needs).

In [ ]:
import json, time
import numpy as np
from marswalk.ml.heuristic import build_dataset, train_mlp, evaluate

N_TERRAINS = 120   # raise for a stronger model
EPOCHS = 120

t0 = time.time()
X, y, flat = build_dataset(range(N_TERRAINS), shape=(40, 40), cellsize=10.0)
print(f'dataset: {len(X):,} cells from {N_TERRAINS} terrains')
model = train_mlp(X, y, epochs=EPOCHS, device=dev)
rep = evaluate(model, X, y, flat)
secs = time.time() - t0
print(rep)

In [ ]:
report = {**rep.as_dict(), 'device': (torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU'),
          'n_terrains': N_TERRAINS, 'epochs': EPOCHS, 'seconds': round(secs, 1),
          'final_loss': round(model.history[-1], 5)}
with open('heuristic_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('wrote heuristic_report.json')
print(json.dumps(report, indent=2))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pred = model.predict(X)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
idx = np.random.default_rng(0).choice(len(y), size=min(4000, len(y)), replace=False)
ax[0].scatter(y[idx]/60, pred[idx]/60, s=4, alpha=.3, color='#e2572b')
lim = max(y[idx].max(), pred[idx].max())/60
ax[0].plot([0, lim], [0, lim], 'k--', lw=1)
ax[0].set_xlabel('true cost-to-go (min)'); ax[0].set_ylabel('predicted (min)')
ax[0].set_title(f"learned heuristic — MAE {rep.mae_s/60:.2f} min, "
                f"{rep.admissible_fraction*100:.1f}% admissible")
ax[1].plot(np.array(model.history)); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('loss')
ax[1].set_title('training loss')
fig.tight_layout(); fig.savefig('heuristic_samples.png', dpi=120)
print('wrote heuristic_samples.png')

## Send back

Download **`heuristic_report.json`** and **`heuristic_samples.png`** and send them to the lead, plus your GPU name (printed above). Thanks!